## Knowledge Graph and ETL Pipeline

A medical knowledge graph from scratch: schema design, Cypher queries, Batch ingestion, LLM-powered entity extraction

In [19]:
import os
import sys
from dotenv import load_dotenv

sys.path.append('..')
from utils import get_neo4j_client

load_dotenv()

True

### Connect to Neo4j

In [20]:
client = get_neo4j_client()
client.verify_connectivity()

2026-04-05 21:17:46.023 | SUCCESS  | utils.neo4j_client:verify_connectivity:22 - Connected to Neo4j at neo4j+s://addafab8.databases.neo4j.io


True

## Graph Schema Design

Drug Interactions: 4 entity types and 5 relationship types

In [21]:
NODE_LABELS = {
    "Drug": "A pharmacological agent (e.g., Warfarin, Aspirin)",
    "Condition": "A medical condition or disease (e.g., Diabetes, Hypertension)",
    "SideEffect": "An adverse effect caused by a drug (e.g., Bleeding, Nausea)",
    "DrugClass": "A pharmacological class (e.g., NSAID, Anticoagulant, SSRI)",
}

RELATIONSHIP_TYPES = {
    "INTERACTS_WITH": "Drug-Drug interaction (bidirectional in meaning, stored as directed edge)",
    "TREATS": "Drug treats a Condition",
    "CAUSES_SIDE_EFFECT": "Drug causes a SideEffect",
    "CONTRAINDICATED_FOR": "Drug is contraindicated for a Condition",
    "BELONGS_TO_CLASS": "Drug belongs to a DrugClass",
}

ALLOWED_REL_TYPES = set(RELATIONSHIP_TYPES.keys())

print("Node Labels:")
for label, desc in NODE_LABELS.items():
    print(f"  :{label} - {desc}")

print("\nRelationship Types:")
for rel_type, desc in RELATIONSHIP_TYPES.items():
    print(f"  [{rel_type}] - {desc}")

Node Labels:
  :Drug - A pharmacological agent (e.g., Warfarin, Aspirin)
  :Condition - A medical condition or disease (e.g., Diabetes, Hypertension)
  :SideEffect - An adverse effect caused by a drug (e.g., Bleeding, Nausea)
  :DrugClass - A pharmacological class (e.g., NSAID, Anticoagulant, SSRI)

Relationship Types:
  [INTERACTS_WITH] - Drug-Drug interaction (bidirectional in meaning, stored as directed edge)
  [TREATS] - Drug treats a Condition
  [CAUSES_SIDE_EFFECT] - Drug causes a SideEffect
  [CONTRAINDICATED_FOR] - Drug is contraindicated for a Condition
  [BELONGS_TO_CLASS] - Drug belongs to a DrugClass


### Constraints, Indexes & Sample Data

CONSTRAINT_QUERIES

- ```CREATE CONSTRAINT ... IF NOT EXISTS``` means “create this constraint only if it hasn’t already been created.”
- ```FOR (d:Drug)``` targets all nodes with the label ```Drug```.
- ```REQUIRE d.name IS UNIQUE``` enforces that the ```name``` property must be unique across all ```Drug``` nodes.

So practically, it prevents duplicate drug names like two separate Drug nodes both having ```name = "Warfarin```".

In [26]:
CONSTRAINT_QUERIES = [
    "CREATE CONSTRAINT drug_name IF NOT EXISTS FOR (d:Drug) REQUIRE d.name IS UNIQUE",
    "CREATE CONSTRAINT condition_name IF NOT EXISTS FOR (c:Condition) REQUIRE c.name IS UNIQUE",
    "CREATE CONSTRAINT side_effect_name IF NOT EXISTS FOR (s:SideEffect) REQUIRE s.name IS UNIQUE",
    "CREATE CONSTRAINT drug_class_name IF NOT EXISTS FOR (dc:DrugClass) REQUIRE dc.name IS UNIQUE",
]

INDEX QUERIES

- ```CREATE INDEX ... IF NOT EXISTS``` means create it only if it is not already present.
- ```FOR (d:Drug)``` says the index applies to nodes with the ```Drug``` label.
- ```ON (d.name)``` means it indexes the name property of those ```Drug``` nodes.

Why it helps: it speeds up lookups and filters like ```MATCH (d:Drug {name: 'Warfarin'})``` by avoiding full scans of all Drug nodes.

In [27]:
INDEX_QUERIES = [
    "CREATE INDEX drug_name_idx IF NOT EXISTS FOR (d:Drug) ON (d.name)",
    "CREATE INDEX condition_name_idx IF NOT EXISTS FOR (c:Condition) ON (c.name)",
    "CREATE INDEX side_effect_name_idx IF NOT EXISTS FOR (s:SideEffect) ON (s.name)",
    "CREATE INDEX drug_class_name_idx IF NOT EXISTS FOR (dc:DrugClass) ON (dc.name)",
]

In [28]:
# To clear the Graph database, run the following query:
client.write("MATCH (n) DETACH DELETE n")

In [33]:
# # Cleanup indexes
# client.write("DROP INDEX drug_name_idx IF EXISTS")
# client.write("DROP INDEX condition_name_idx IF EXISTS")
# client.write("DROP INDEX side_effect_name_idx IF EXISTS")
# client.write("DROP INDEX drug_class_name_idx IF EXISTS")

In [34]:
# Create constraints and indexes
for q in CONSTRAINT_QUERIES + INDEX_QUERIES:
    try:
        client.write(q)
        print(f"Executed: {q}")
    except Exception as e:
        print(f"Error executing {q}: {e}")

print("Schema created successfully.")

Executed: CREATE CONSTRAINT drug_name IF NOT EXISTS FOR (d:Drug) REQUIRE d.name IS UNIQUE
Executed: CREATE CONSTRAINT condition_name IF NOT EXISTS FOR (c:Condition) REQUIRE c.name IS UNIQUE
Executed: CREATE CONSTRAINT side_effect_name IF NOT EXISTS FOR (s:SideEffect) REQUIRE s.name IS UNIQUE
Executed: CREATE CONSTRAINT drug_class_name IF NOT EXISTS FOR (dc:DrugClass) REQUIRE dc.name IS UNIQUE
Executed: CREATE INDEX drug_name_idx IF NOT EXISTS FOR (d:Drug) ON (d.name)
Executed: CREATE INDEX condition_name_idx IF NOT EXISTS FOR (c:Condition) ON (c.name)
Executed: CREATE INDEX side_effect_name_idx IF NOT EXISTS FOR (s:SideEffect) ON (s.name)
Executed: CREATE INDEX drug_class_name_idx IF NOT EXISTS FOR (dc:DrugClass) ON (dc.name)
Schema created successfully.


In [35]:
# Create a sample drugs, conditions, relationships
sample_cypher = """
CREATE (warfarin:Drug {name: 'Warfarin', description: 'Anticoagulant medication'})
CREATE (aspirin:Drug {name: 'Aspirin', description: 'NSAID and antiplatelet agent'})
CREATE (ibuprofen:Drug {name: 'Ibuprofen', description: 'Non-steroidal anti-inflammatory drug'})
CREATE (bleeding:SideEffect {name: 'Increased Bleeding Risk'})
CREATE (dvt:Condition {name: 'Deep Vein Thrombosis'})
CREATE (anticoag:DrugClass {name: 'Anticoagulant'})
CREATE (nsaid:DrugClass {name: 'NSAID'})

CREATE (warfarin)-[:INTERACTS_WITH {severity: 'Major', mechanism: 'Additive anticoagulant effect'}]->(aspirin)
CREATE (warfarin)-[:INTERACTS_WITH {severity: 'Major', mechanism: 'Increased bleeding via platelet and coagulation pathway'}]->(ibuprofen)
CREATE (warfarin)-[:CAUSES_SIDE_EFFECT]->(bleeding)
CREATE (aspirin)-[:CAUSES_SIDE_EFFECT]->(bleeding)
CREATE (warfarin)-[:TREATS]->(dvt)
CREATE (warfarin)-[:BELONGS_TO_CLASS]->(anticoag)
CREATE (aspirin)-[:BELONGS_TO_CLASS]->(nsaid)
CREATE (ibuprofen)-[:BELONGS_TO_CLASS]->(nsaid)
"""

client.write(sample_cypher)
print("Sample data created successfully.")

Sample data created successfully.


> **Try in Neo4j Browser**:
>
> ```cypher
> -- See all nodes and relationships
> MATCH (n)-[r]->(m) RETURN n, r, m
> ```
>
> ```cypher
> -- Check constraints
> SHOW CONSTRAINTS
> ```
>
> ```cypher
> -- Check indexes
> SHOW INDEXES
> ```